# A6 Keyword & TF-IDF Baselines (CPU, LOCKED TEST)

Melatih baseline keyword + TF-IDF pada train, memilih pada validation, lalu
mengevaluasi locked test **sekali** lewat `sipature_ml.baselines.run_baselines`.
Ikuti `docs/leakage-safe-split-baseline-report.md` sebelum eksekusi.

Input: `data/splits/*` (dari notebook `04`). Output: `artifacts/metrics/`,
`artifacts/models/`, `artifacts/reports/baseline_*`, dan 3 figure.

**PENTING:** notebook ini MEMBACA locked test untuk evaluasi satu-kali. Metric
tidak boleh ditimpa — `run_baselines` menolak jika metric sudah ada. Hasil adalah
*agreement terhadap silver labels*, bukan human-gold accuracy.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_SPLIT_DIR = DRIVE_ROOT / "data" / "splits"

PROJECT_DIR = Path("/content/hackathon/ml")
SPLIT_DIR = PROJECT_DIR / "data" / "splits"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
FIGURE_DIR = ARTIFACT_DIR / "figures" / "baselines"

DRIVE_METRICS_DIR = DRIVE_ROOT / "metrics"
DRIVE_MODELS_DIR = DRIVE_ROOT / "models"
DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
DRIVE_FIGURE_DIR = DRIVE_ROOT / "figures" / "baselines"

print("Drive root:", DRIVE_ROOT)
print("Sumber split:", DRIVE_SPLIT_DIR)
print("Artifact dir (lokal):", ARTIFACT_DIR)
print("Figure dir (lokal)  :", FIGURE_DIR)


In [ ]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


In [ ]:
%cd /content/hackathon/ml
!git log --oneline -3


In [ ]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


In [ ]:
import numpy
import pandas
import pyarrow
import sklearn
import joblib
import matplotlib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)
print("Matplotlib:", matplotlib.__version__)


In [ ]:
# Salin split (train/validation/test + manifest) dari Drive ke lokal.
import shutil
from pathlib import Path

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

split_files = [
    "train_silver_v1.jsonl",
    "validation_silver_v1.jsonl",
    "test_silver_v1.jsonl",
    "split_manifest_silver_v1.json",
]

for filename in split_files:
    source = DRIVE_SPLIT_DIR / filename
    assert source.is_file(), f"Split file tidak ditemukan di Drive: {source}"
    shutil.copy2(source, SPLIT_DIR / filename)
    print("Disalin:", filename)


In [ ]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


In [ ]:
import json
from sipature_ml.config import load_config

config = load_config("training")

manifest = json.loads((SPLIT_DIR / "split_manifest_silver_v1.json").read_text(encoding="utf-8"))
assert manifest.get("test_is_locked"), "Manifest split tidak terkunci (test_is_locked != true)"

print("Experiment version:", config["experiment_version"])
print("Seed:", config["seed"])
print("Task:", config["task"])
print("Split version:", manifest["split_version"])
print("Test is locked:", manifest["test_is_locked"])
print("TF-IDF representations:", config["baselines"]["tfidf"]["representations"])
print("TF-IDF C values:", config["baselines"]["tfidf"]["c_values"])


In [ ]:
# Guard: evaluasi locked test hanya boleh sekali.
import shutil
from pathlib import Path

drive_metrics_guard = DRIVE_METRICS_DIR / "keyword-silver-v1-test-metrics.json"

if drive_metrics_guard.is_file():
    print("Baseline metric SUDAH ada di Drive. Menyalin hasil yang ada, TANPA evaluasi ulang.")
    for drive_dir, local_dir in (
        (DRIVE_METRICS_DIR, ARTIFACT_DIR / "metrics"),
        (DRIVE_MODELS_DIR, ARTIFACT_DIR / "models"),
        (DRIVE_REPORT_DIR, ARTIFACT_DIR / "reports"),
        (DRIVE_FIGURE_DIR, FIGURE_DIR),
    ):
        local_dir.mkdir(parents=True, exist_ok=True)
        for source in sorted(drive_dir.glob("*")):
            if source.is_file():
                shutil.copy2(source, local_dir / source.name)
                print(f"Disalin dari Drive: {source.name} -> {local_dir}")
else:
    from sipature_ml.baselines import run_baselines
    summary = run_baselines(SPLIT_DIR, ARTIFACT_DIR, FIGURE_DIR)
    print("Baseline berhasil dilatih & dievaluasi pada locked test.")
    print("Keyword Macro F1:", round(summary["keyword_test_macro_f1"], 4))
    print("TF-IDF  Macro F1:", round(summary["tfidf_test_macro_f1"], 4))
    print("Keyword Micro F1:", round(summary["keyword_test_micro_f1"], 4))
    print("TF-IDF  Micro F1:", round(summary["tfidf_test_micro_f1"], 4))


In [ ]:
# Tampilkan metric locked-test (dari file yang baru/telah dibuat).
import json
from pathlib import Path

for name in ("keyword-silver-v1-test-metrics.json", "tfidf-silver-v1-test-metrics.json"):
    path = ARTIFACT_DIR / "metrics" / name
    if not path.is_file():
        print(f"(skip, belum ada) {name}")
        continue
    metrics = json.loads(path.read_text(encoding="utf-8"))
    print(f"=== {name} ===")
    print("  Macro F1:", round(metrics["macro_f1"], 4))
    print("  Micro F1:", round(metrics["micro_f1"], 4))
    print("  Exact Match:", round(metrics["exact_match"], 4))
    print("  Hamming Loss:", round(metrics["hamming_loss"], 4))
    print("  Latency ms/review:", round(metrics["latency_ms_per_review"], 4))


In [ ]:
# Salin metrics/models/reports/figure ke Drive (artefak persisten).
import shutil
from pathlib import Path

for local_dir, drive_dir in (
    (ARTIFACT_DIR / "metrics", DRIVE_METRICS_DIR),
    (ARTIFACT_DIR / "models", DRIVE_MODELS_DIR),
    (ARTIFACT_DIR / "reports", DRIVE_REPORT_DIR),
    (FIGURE_DIR, DRIVE_FIGURE_DIR),
):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for source in sorted(local_dir.glob("*")):
        if source.is_file():
            shutil.copy2(source, drive_dir / source.name)
            print(f"Disalin: {source.name} -> {drive_dir}")


In [ ]:
# ============================================================
# RUN SUMMARY — hash, metric, dan limitations.
# ============================================================
import json
from pathlib import Path
from sipature_ml.manifest import sha256_file

baseline_summary = json.loads(
    (ARTIFACT_DIR / "reports" / "baseline_summary.json").read_text(encoding="utf-8")
)

print("EXPERIMENT VERSION:", baseline_summary["experiment_version"])
print("SPLIT VERSION      :", baseline_summary["split_version"])
print("TF-IDF representation:", baseline_summary["selected_tfidf_representation"])
print("Keyword test Macro F1:", round(baseline_summary["keyword_test_macro_f1"], 4))
print("TF-IDF  test Macro F1:", round(baseline_summary["tfidf_test_macro_f1"], 4))
print("Figures:", baseline_summary["figures"])

print("\nOUTPUT METRICS DIR :", ARTIFACT_DIR / "metrics")
print("OUTPUT MODELS DIR  :", ARTIFACT_DIR / "models")
print("OUTPUT REPORTS DIR :", ARTIFACT_DIR / "reports")
print("OUTPUT FIGURE DIR  :", FIGURE_DIR)

print("\nREMINDER: metric ini adalah agreement terhadap silver (weak supervision),")
print("bukan akurasi human-gold. Keyword F1 tinggi bersifat circular terhadap silver rules.")
